# Topic modelling

In [6]:
import sys
sys.path.append('/home/onyxia/work/nlp_central_banks/lyna_work')

from module import *

#### Creating a BERTopic object, fit and transform

Maintenant on va diviser la base de données en chunk, pour pouvoir la traiter avec un modèle avec un contexte window petit.

In [22]:
import sys
print(sys.executable)  # doit afficher .../venv_gpu/bin/python

import torch
print(torch.cuda.is_available())  # doit afficher True
print(torch.cuda.get_device_name(0))  # NVIDIA A2

/home/onyxia/work/venv_gpu/bin/python
True
NVIDIA A2


In [3]:
import os
import re
import spacy
import torch
import pandas as pd
import nltk
from transformers import AutoTokenizer
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords
from tqdm import tqdm

nltk.download('stopwords')

/home/onyxia/work/venv_gpu/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
# ── 0. Vérification GPU ────────────────────────────────────
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


PyTorch  : 2.6.0+cu124
CUDA     : True
GPU      : NVIDIA A2
VRAM     : 15.7 GB


In [25]:
# ── 2. Chunking ────────────────────────────────────────────
if torch.cuda.is_available():
    spacy.prefer_gpu()  # utilise GPU si CuPy dispo, sinon CPU sans erreur

nlp = spacy.load("en_core_web_sm")

tokenizer = AutoTokenizer.from_pretrained("hugom123/bge-centralbank")

def count_tokens_batch(sentences, batch_size=512):
    all_counts = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i + batch_size]
        encoded = tokenizer(
            batch,
            add_special_tokens=False,
            return_length=True,
            truncation=False,
        )
        all_counts.extend(encoded['length'])
    return all_counts


def process_corpus(df, max_tokens=450, overlap_sents=2, batch_size=64):
    records = []
    texts   = df['text_clean'].tolist()
    indices = df.index.tolist()

    print("Segmentation en phrases (spaCy)...")
    docs = list(tqdm(
        nlp.pipe(texts, batch_size=batch_size),
        total=len(texts),
        desc="spaCy"
    ))

    print("Chunking...")
    for doc, idx in tqdm(zip(docs, indices), total=len(docs), desc="Chunking"):
        row = df.loc[idx]
        sentences = [s.text.strip() for s in doc.sents if s.text.strip()]
        if not sentences:
            continue

        token_counts = count_tokens_batch(sentences)
        chunks, current_sents, current_tokens = [], [], 0

        for sent, n_tok in zip(sentences, token_counts):
            if n_tok > max_tokens:
                if current_sents:
                    chunks.append(" ".join(current_sents))
                chunks.append(sent)
                current_sents, current_tokens = [], 0
                continue

            if current_tokens + n_tok > max_tokens and current_sents:
                chunks.append(" ".join(current_sents))
                current_sents = current_sents[-overlap_sents:]
                current_tokens = sum(
                    count_tokens_batch([s])[0] for s in current_sents
                )

            current_sents.append(sent)
            current_tokens += n_tok

        if current_sents:
            chunks.append(" ".join(current_sents))

        for i, chunk in enumerate(chunks):
            records.append({
                'doc_id':      row['Unnamed: 0'],
                'chunk_id':    i,
                'CentralBank': row['CentralBank'],
                'Date':        row['Date'],
                'Year':        row['Year'],
                'Authorname':  row['Authorname'],
                'Role':        row['Role'],
                'Source':      row['Source'],
                'chunk_text':  chunk,
                'n_words':     len(chunk.split()),
            })

    return pd.DataFrame(records)


df_chunks = process_corpus(df_filtered)

print(f"\nDiscours        : {len(df_filtered)}")
print(f"Chunks          : {len(df_chunks)}")
print(f"Chunks/discours : {len(df_chunks)/len(df_filtered):.1f}")
print(f"\nTaille chunks (mots) :")
print(df_chunks['n_words'].describe().round(0))
print(f"\nChunks < 50 mots : {(df_chunks['n_words'] < 50).sum()}")


Segmentation en phrases (spaCy)...


spaCy: 100%|██████████| 2286/2286 [13:53<00:00,  2.74it/s]


Chunking...


Chunking: 100%|██████████| 2286/2286 [00:26<00:00, 86.06it/s] 



Discours        : 2286
Chunks          : 23913
Chunks/discours : 10.5

Taille chunks (mots) :
count    23913.0
mean       342.0
std         58.0
min         18.0
25%        339.0
50%        359.0
75%        372.0
max        477.0
Name: n_words, dtype: float64

Chunks < 50 mots : 42


In [26]:
# Si tu as un bucket S3 configuré sur Onyxia
import s3fs
df_chunks.to_csv("s3://lelkamel/chunks/df_chunks_all.csv", index=False)
df_stratified.to_csv("s3://lelkamel/chunks/df_all_clean.csv", index=False)

In [27]:
import s3fs
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
import io

# Connexion S3 Onyxia
fs = s3fs.S3FileSystem(
    client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"}
)

# Vérifier ce qui est disponible
print("Fichiers disponibles :")
print(fs.ls("lelkamel/chunks/"))

# Réimporter df_chunks DE TOUTE LA BASE DE DONNEES !!!!!!!!!!!!!!!!!!!!!
print("\nChargement df_chunks...")
with fs.open("lelkamel/chunks/df_chunks_all.csv", "rb") as f:
    df_chunks = pd.read_csv(f)
print(f"df_chunks chargé : {df_chunks.shape}")

# Définir docs
docs = df_chunks['chunk_text'].tolist()
print(f"Nombre de chunks : {len(docs)}")

# Recalculer embeddings sur GPU
print("\nChargement du modèle...")
embedding_model = SentenceTransformer("hugom123/bge-centralbank")

print("Calcul des embeddings...")
embeddings = embedding_model.encode(
    docs,
    batch_size=64,
    show_progress_bar=True,
    device="cuda" if torch.cuda.is_available() else "cpu"
)
print(f"Embeddings shape : {embeddings.shape}")

# Sauvegarder immédiatement sur S3
print("\nSauvegarde embeddings sur S3...")
with fs.open("lelkamel/chunks/embeddings_all.npy", "wb") as f:
    np.save(f, embeddings)
print("Embeddings sauvegardés : s3://lelkamel/chunks/embeddings_all.npy")

Fichiers disponibles :
['lelkamel/chunks/df_all_clean.csv', 'lelkamel/chunks/df_chunks.csv', 'lelkamel/chunks/df_chunks_all.csv', 'lelkamel/chunks/df_stratified_clean.csv', 'lelkamel/chunks/embeddings.npy']

Chargement df_chunks...
df_chunks chargé : (23913, 10)
Nombre de chunks : 23913

Chargement du modèle...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2917.10it/s]


Calcul des embeddings...


Batches: 100%|██████████| 374/374 [33:56<00:00,  5.45s/it]


Embeddings shape : (23913, 1024)

Sauvegarde embeddings sur S3...
Embeddings sauvegardés : s3://lelkamel/chunks/embeddings_all.npy


###### Analyse stratifiée

Now we choose the embedding model. On the Hugging face, I found several models that might be interesting : *hugom123/bge-centralbank* which is a sentence embedder trained on central bank speeches, and others such as ... Let us now check if our documents fit inside the context window. We retrieve that information from the HuggingFace page of the model.

In [26]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("hugom123/bge-centralbank")
print(config.max_position_embeddings)

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


512


In [ ]:
# ── 2. Chunking ────────────────────────────────────────────
if torch.cuda.is_available():
    spacy.prefer_gpu()  # utilise GPU si CuPy dispo, sinon CPU sans erreur

nlp = spacy.load("en_core_web_sm")

tokenizer = AutoTokenizer.from_pretrained("hugom123/bge-centralbank")

def count_tokens_batch(sentences, batch_size=512):
    all_counts = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i + batch_size]
        encoded = tokenizer(
            batch,
            add_special_tokens=False,
            return_length=True,
            truncation=False,
        )
        all_counts.extend(encoded['length'])
    return all_counts


def process_corpus(df, max_tokens=450, overlap_sents=2, batch_size=64):
    records = []
    texts   = df['text_clean'].tolist()
    indices = df.index.tolist()

    print("Segmentation en phrases (spaCy)...")
    docs = list(tqdm(
        nlp.pipe(texts, batch_size=batch_size),
        total=len(texts),
        desc="spaCy"
    ))

    print("Chunking...")
    for doc, idx in tqdm(zip(docs, indices), total=len(docs), desc="Chunking"):
        row = df.loc[idx]
        sentences = [s.text.strip() for s in doc.sents if s.text.strip()]
        if not sentences:
            continue

        token_counts = count_tokens_batch(sentences)
        chunks, current_sents, current_tokens = [], [], 0

        for sent, n_tok in zip(sentences, token_counts):
            if n_tok > max_tokens:
                if current_sents:
                    chunks.append(" ".join(current_sents))
                chunks.append(sent)
                current_sents, current_tokens = [], 0
                continue

            if current_tokens + n_tok > max_tokens and current_sents:
                chunks.append(" ".join(current_sents))
                current_sents = current_sents[-overlap_sents:]
                current_tokens = sum(
                    count_tokens_batch([s])[0] for s in current_sents
                )

            current_sents.append(sent)
            current_tokens += n_tok

        if current_sents:
            chunks.append(" ".join(current_sents))

        for i, chunk in enumerate(chunks):
            records.append({
                'doc_id':      row['Unnamed: 0'],
                'chunk_id':    i,
                'CentralBank': row['CentralBank'],
                'Date':        row['Date'],
                'Year':        row['Year'],
                'Authorname':  row['Authorname'],
                'Role':        row['Role'],
                'Source':      row['Source'],
                'chunk_text':  chunk,
                'n_words':     len(chunk.split()),
            })

    return pd.DataFrame(records)


df_chunks = process_corpus(df_stratified)

print(f"\nDiscours        : {len(df_stratified)}")
print(f"Chunks          : {len(df_chunks)}")
print(f"Chunks/discours : {len(df_chunks)/len(df_stratified):.1f}")
print(f"\nTaille chunks (mots) :")
print(df_chunks['n_words'].describe().round(0))
print(f"\nChunks < 50 mots : {(df_chunks['n_words'] < 50).sum()}")


Segmentation en phrases (spaCy)...


spaCy: 100%|██████████| 483/483 [02:35<00:00,  3.11it/s]


Chunking...


Chunking: 100%|██████████| 483/483 [00:06<00:00, 74.86it/s] 


Discours        : 483
Chunks          : 4941
Chunks/discours : 10.2

Taille chunks (mots) :
count    4941.0
mean      343.0
std        58.0
min        32.0
25%       341.0
50%       360.0
75%       374.0
max       410.0
Name: n_words, dtype: float64

Chunks < 50 mots : 8


In [30]:
# Si tu as un bucket S3 configuré sur Onyxia
import s3fs
df_chunks.to_csv("s3://lelkamel/chunks/df_chunks.csv", index=False)
df_stratified.to_csv("s3://lelkamel/chunks/df_stratified_clean.csv", index=False)

In [5]:
# ── 3. Stopwords et Vectorizer ─────────────────────────────
standard_stopwords = list(stopwords.words('english'))

additional_stopwords = [
    # Institution names
    "ecb", "eurosystem", "eurozone", "fed", "federal", "reserve",
    "fomc", "boe", "bank", "banks", "central", "committee",
    "council", "governing", "board",
    
    # Banques centrales en toutes lettres
    "european", "england", "governors",
    
    # Institutions internationales
    "european union", "european commission", "european parliament",
    "imf", "international monetary fund",
    "bis", "bank international settlements",
    "oecd", "world bank", "g7", "g20",
    "united nations", "wto",
    
    # Pays majeurs fréquemment cités
    "united states", "america", "american",
    "germany", "german", "france", "french",
    "italy", "italian", "spain", "spanish",
    "japan", "japanese", "china", "chinese",
    "uk", "britain", "british",
    "greece", "greek", "portugal", "portuguese",
    "ireland", "irish", "netherlands", "dutch",
    "switzerland", "swiss", "sweden", "swedish",
    "denmark", "danish", "norway", "norwegian",
    "canada", "canadian", "australia", "australian",
    
    # Zones géographiques génériques
    "euro", "area", "zone", "region", "regions",
    "global", "international", "domestic",
    "world", "worldwide", "cross", "border",
    
    # Personnes et rôles
    "president", "governor", "chairman", "vice", "deputy",
    "mr", "mrs", "dr", "professor", "minister",
    "government", "governments",
    
    # Speech fillers
    "today", "thank", "thanks", "pleasure", "honored", "welcome",
    "morning", "afternoon", "evening", "conference", "symposium",
    "ladies", "gentlemen", "colleagues", "friends",
    "speech", "remarks", "address", "statement", "introductory",
    "keynote", "presentation", "lecture",
    
    # Generic time words
    "year", "years", "month", "months", "day", "days",
    "time", "times", "period", "periods",
    "recent", "recently", "current", "currently",
    "past", "future", "last", "next", "first", "second",
    "quarter", "quarters", "decade", "decades",
    
    # Termes macro génériques
    "economy", "economic", "economies",
    "growth", "rate", "rates",
    "policy", "policies",
    "market", "markets",
    "percent", "percentage",
    "billion", "trillion", "million",
    
    # Discourse fillers
    "also", "however", "therefore", "thus", "indeed",
    "moreover", "furthermore", "course", "fact", "example",
    "important", "significant", "particular", "general",
    "well", "said", "say", "know", "think",
    "need", "want", "make", "take", "come", "going",
    "one", "two", "three", "many", "much", "large", "small",
    
    # Termes juridiques/institutionnels génériques
    "act", "law", "regulation", "framework", "mandate",
    "article", "treaty", "agreement",
     # Community reinvestment (spécifique Fed)
    "community", "communities", "community development",
    "reinvestment", "cra", "neighborhood", "neighborhoods",
    
    # Termes trop génériques qui polluent topic 0
    "financial", "monetary", "stability",
    "would", "could", "should", "may", "might",
    "also", "including", "however",
    
    # Banknotes/coins (topic -1)
    "banknotes", "coins", "cash", "currency",
      "inflation", "risk", "capital", "interest", "system",
    "new", "price", "stability", "crisis", "debt",
    "support", "measures", "asset", "purchases",
]

full_stopwords = standard_stopwords + additional_stopwords

vectorizer_model = CountVectorizer(
    stop_words=full_stopwords,
    ngram_range=(1, 2),
    min_df=2,
    lowercase=True
)

In [8]:
import s3fs
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
import io

# Connexion S3 Onyxia
fs = s3fs.S3FileSystem(
    client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"}
)

# Vérifier ce qui est disponible
print("Fichiers disponibles :")
print(fs.ls("lelkamel/chunks/"))

# Réimporter df_chunks
print("\nChargement df_chunks...")
with fs.open("lelkamel/chunks/df_chunks.csv", "rb") as f:
    df_chunks = pd.read_csv(f)
print(f"df_chunks chargé : {df_chunks.shape}")

# Définir docs
docs = df_chunks['chunk_text'].tolist()
print(f"Nombre de chunks : {len(docs)}")

# Recalculer embeddings sur GPU
print("\nChargement du modèle...")
embedding_model = SentenceTransformer("hugom123/bge-centralbank")

print("Calcul des embeddings...")
embeddings = embedding_model.encode(
    docs,
    batch_size=64,
    show_progress_bar=True,
    device="cuda" if torch.cuda.is_available() else "cpu"
)
print(f"Embeddings shape : {embeddings.shape}")

# Sauvegarder immédiatement sur S3
print("\nSauvegarde embeddings sur S3...")
with fs.open("lelkamel/chunks/embeddings.npy", "wb") as f:
    np.save(f, embeddings)
print("Embeddings sauvegardés : s3://lelkamel/chunks/embeddings.npy")

Fichiers disponibles :
['lelkamel/chunks/df_chunks.csv', 'lelkamel/chunks/df_stratified_clean.csv']

Chargement df_chunks...


df_chunks chargé : (4941, 10)
Nombre de chunks : 4941

Chargement du modèle...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2902.43it/s]


Calcul des embeddings...


Batches: 100%|██████████| 78/78 [07:06<00:00,  5.46s/it]


Embeddings shape : (4941, 1024)

Sauvegarde embeddings sur S3...
Embeddings sauvegardés : s3://lelkamel/chunks/embeddings.npy


# All data

In [29]:
# ── 4. BERTopic ────────────────────────────────────────────
docs = df_chunks['chunk_text'].tolist()
#topic_model = BERTopic(
#    language="english",
#    embedding_model="hugom123/bge-centralbank",
#    vectorizer_model=vectorizer_model,
#    min_topic_size=30,
#    nr_topics= None,
#    calculate_probabilities=True,
#    verbose=True,
#)
from umap import UMAP

RANDOM_SEED = 42

default_umap_parameters = {
    "n_neighbors": 15,
    "n_components": 5,
    "min_dist": 0.0,
    "metric": "cosine",
}

topic_model = BERTopic(
    language="english",
    embedding_model="hugom123/bge-centralbank",
    vectorizer_model=vectorizer_model,
    umap_model=UMAP(**default_umap_parameters, random_state=RANDOM_SEED),
    min_topic_size=30,
    nr_topics=None,
    calculate_probabilities=True,
    verbose=True,
)

topics, probs = topic_model.fit_transform(documents=docs,
    embeddings=embeddings)

# ── Résultats ──────────────────────────────────────────────
df_chunks['topic']      = topics
df_chunks['topic_prob'] = probs.max(axis=1) if probs.ndim > 1 else probs

print(f"\nNombre de topics : {topic_model.get_topic_info().shape[0] - 1}")
print("\n=== Top 20 topics ===")
print(topic_model.get_topic_info().head(20))

# ── Sauvegarde immédiate ───────────────────────────────────
fs = s3fs.S3FileSystem(
    client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"}
)

with fs.open("lelkamel/chunks/df_chunks_with_topics.csv", "wb") as f:
    df_chunks.to_csv(f, index=False)
print("df_chunks_with_topics sauvegardé sur S3 !")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3798.06it/s]
2026-04-27 15:22:57,496 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-27 15:23:55,643 - BERTopic - Dimensionality - Completed ✓
2026-04-27 15:23:55,646 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-27 15:24:07,112 - BERTopic - Cluster - Completed ✓
2026-04-27 15:24:07,124 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-27 15:24:21,728 - BERTopic - Representation - Completed ✓



Nombre de topics : 114

=== Top 20 topics ===
    Topic  Count                                                       Name  \
0      -1  10021                                -1_term_countries_credit_us   
1       0    876                         0_mortgage_housing_house_mortgages   
2       1    551                                  1_mpc_demand_brexit_chart   
3       2    443        2_macroprudential_macro_macro prudential_prudential   
4       3    430  3_liquidity_operations_refinancing_refinancing operations   
5       4    399                             4_fiscal_pact_countries_public   
6       5    294                      5_integration_single_integrated_union   
7       6    292                            6_union_europe_single_political   
8       7    287                              7_money_smith_economics_notes   
9       8    280                          8_medium_medium term_term_outlook   
10      9    278                        9_liquidity_collateral_assets_funds   
11   

In [8]:
import s3fs
import pandas as pd
import numpy as np

# Connexion S3 Onyxia
fs = s3fs.S3FileSystem(
    client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"}
)

# Vérifier les fichiers disponibles
print("Fichiers disponibles :")
print(fs.ls("lelkamel/chunks/"))

# Importer df_chunks
print("\nChargement df_chunks...")
with fs.open("lelkamel/chunks/df_chunks_all.csv", "rb") as f:
    df_chunks = pd.read_csv(f)
print(f"df_chunks chargé : {df_chunks.shape}")

# Définir docs
docs = df_chunks['chunk_text'].tolist()
print(f"Nombre de chunks : {len(docs)}")

# Importer embeddings
print("\nChargement embeddings...")
with fs.open("lelkamel/chunks/embeddings_all.npy", "rb") as f:
    embeddings = np.load(f)
print(f"Embeddings chargés : {embeddings.shape}")

Fichiers disponibles :
['lelkamel/chunks/df_all_clean.csv', 'lelkamel/chunks/df_chunks.csv', 'lelkamel/chunks/df_chunks_all.csv', 'lelkamel/chunks/df_chunks_with_topics.csv', 'lelkamel/chunks/df_stratified_clean.csv', 'lelkamel/chunks/embeddings.npy', 'lelkamel/chunks/embeddings_all.npy']

Chargement df_chunks...
df_chunks chargé : (23913, 10)
Nombre de chunks : 23913

Chargement embeddings...
Embeddings chargés : (23913, 1024)


In [9]:
from umap import UMAP

RANDOM_SEED = 42

default_umap_parameters = {
    "n_neighbors": 15,
    "n_components": 5,
    "min_dist": 0.0,
    "metric": "cosine",
}

from hdbscan import HDBSCAN

hdbscan_model = HDBSCAN(
    min_cluster_size=20,      # plus petit = plus de topics
    min_samples=5,            # moins strict = moins d'outliers
    metric='euclidean',
    cluster_selection_method='leaf',  # 'leaf' crée plus de petits clusters
    prediction_data=True
)

topic_model = BERTopic(
    language="english",
    embedding_model="hugom123/bge-centralbank",
    vectorizer_model=vectorizer_model,
    umap_model=UMAP(**default_umap_parameters, random_state=42),
    hdbscan_model=hdbscan_model,
    min_topic_size=20,
    nr_topics=None,
    calculate_probabilities=True,
    verbose=True,
)

topics, probs = topic_model.fit_transform(
    documents=docs,
    embeddings=embeddings
)

print(f"Nombre de topics : {topic_model.get_topic_info().shape[0] - 1}")
print(topic_model.get_topic_info().head(30))

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5674.33it/s]
2026-04-27 16:27:35,406 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-27 16:28:23,418 - BERTopic - Dimensionality - Completed ✓
2026-04-27 16:28:23,421 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-27 16:29:15,026 - BERTopic - Cluster - Completed ✓
2026-04-27 16:29:15,037 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-27 16:29:24,988 - BERTopic - Representation - Completed ✓


Nombre de topics : 275
    Topic  Count                                               Name  \
0      -1  10565                   -1_credit_term_banking_countries   
1       0    191                        0_budget_fiscal_tax_deficit   
2       1    155             1_resolution_creditors_insolvency_bail   
3       2    151                      2_bubble_bubbles_prices_stock   
4       3    144                      3_housing_homes_home_mortgage   
5       4    137           4_account_deficit_saving_account deficit   
6       5    134              5_recovery_parliament_affairs_hearing   
7       6    132        6_facility_facilities_loans_discount window   
8       7    126                7_education_college_students_school   
9       8    124     8_funds_target range_range funds_accommodation   
10      9    122        9_independence_mpc_accountability_decisions   
11     10    115           10_equilibrium_real_equilibrium real_low   
12     11    114  11_integration_integrated_state inte

In [10]:
topics, probabilities = topic_model.transform(documents=docs, embeddings=embeddings)

2026-04-27 16:29:30,315 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-04-27 16:29:30,477 - BERTopic - Dimensionality - Completed ✓
2026-04-27 16:29:30,478 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-04-27 16:29:31,110 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-04-27 16:30:21,071 - BERTopic - Probabilities - Completed ✓
2026-04-27 16:30:21,072 - BERTopic - Cluster - Completed ✓


In [11]:
topic_info = topic_model.get_topic_info()
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,10565,-1_credit_term_banking_countries,"[credit, term, banking, countries, liquidity, ...",[The Bank of Japan has acquired a wide range o...
1,0,191,0_budget_fiscal_tax_deficit,"[budget, fiscal, tax, deficit, spending, cbo, ...","[2 This alternative fiscal policy scenario, wh..."
2,1,155,1_resolution_creditors_insolvency_bail,"[resolution, creditors, insolvency, bail, auth...",[We shall need the documentation of bond issue...
3,2,151,2_bubble_bubbles_prices_stock,"[bubble, bubbles, prices, stock, equity, stock...",[But such declines sometimes appear to be the ...
4,3,144,3_housing_homes_home_mortgage,"[housing, homes, home, mortgage, house, house ...","[As you know, the correction in the housing ma..."
...,...,...,...,...,...
271,270,20,270_yields_yield_term yields_term,"[yields, yield, term yields, term, long term, ...","[In the current episode, however, the more-dis..."
272,271,20,271_communication_communicate_trust_strategy,"[communication, communicate, trust, strategy, ...",[Differences in views and practices need not b...
273,272,20,272_macroprudential_macroprudential tools_tool...,"[macroprudential, macroprudential tools, tools...",[9 Brunnermeier and Sannikov (2013). 10 See Ai...
274,273,20,273_balance sheet_sheet_balance_reserves,"[balance sheet, sheet, balance, reserves, size...",[I thought it would be useful this evening to ...


Ok, we see topics that seem very well-defined emerging ! Indeed, some of them seem easily interpretable and are consistent with what one would expect from central bank communication
1) **housing/mortages** (not estonishing, given that the 2008 financial crisis has shed light on the interraction between the housing market and financial crises; we will see further whether this topic is predominent in 2008 onwards)
2) **climate change** (as highlighted by Campiglio article too; Bertopic manages to find some patterns of this theme which confirm their dictionary approach)
78) **inequality, wealth, income** (this is intersting as it is a very clear topic, that isn't historically on the central bank's agenda. We will try to understand more its presence by studying its timing)
79) **lehman, stress, treasury** (this topic refers to the 2008 crisis for sure)

This is very encouraging, though Topic -1 remains very high (31 % of our database is cast as outlier). Now, we will reassign clusters to documents clustered as noise with the *reduce_outlier* method. There are several strategies to do that, I focus on the *embedding* one as embeddings offer the richest representation of chunks (especially as I used an embedding model that was trained on central bank data). This strategy consists in finding the best matching topic using cosine similarity between the outlier document's embedding and the topic centroid. 

In [12]:
new_topics = topic_model.reduce_outliers(
    documents=docs,
    topics=topics,
    probabilities=probs,
    embeddings=embeddings,
    strategy="embeddings",
    threshold=0.0
)

# Mettre à jour le modèle
topic_model.update_topics(
    docs,
    topics=new_topics,
    vectorizer_model=vectorizer_model
)

df_chunks['topic'] = new_topics
print(f"Outliers restants : {(pd.Series(new_topics) == -1).sum()}")
print(topic_model.get_topic_info().head(20))

2026-04-27 16:30:21,600 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Outliers restants : 0
    Topic  Count                                               Name  \
0       0    219                        0_budget_fiscal_tax_deficit   
1       1    189             1_resolution_creditors_insolvency_bail   
2       2    180                      2_bubble_bubbles_prices_stock   
3       3    187                      3_housing_home_mortgage_homes   
4       4    171           4_account_deficit_saving_account deficit   
5       5    289             5_recovery_outlook_parliament_pandemic   
6       6    162                6_facility_facilities_loans_funding   
7       7    130                7_education_college_students_school   
8       8    160           8_funds_target range_outlook_range funds   
9       9    144             9_independence_mpc_decisions_political   
10     10    159                    10_equilibrium_real_natural_low   
11     11    171  11_integration_integrated_single_integration e...   
12     12    115    12_shadow_shadow banking_banking_ba

I do not have any outlier anymore and the topics are coherent and precise. Some of them are very well defined (1. housing market, 2. climate change, 4. labour market, 6. fiscal policy, 7. payment system, 9. financial regulation) and some might be fusionnable (topic 16, 17, 18 on banking regulation, among others).

In [13]:
import s3fs
fs = s3fs.S3FileSystem(
    client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"}
)

# Sauvegarder df_chunks avec topics
with fs.open("lelkamel/chunks/df_chunks_with_topics_all.csv", "wb") as f:
    df_chunks.to_csv(f, index=False)
print("df_chunks_with_topics sauvegardé !")

# Sauvegarder le modèle BERTopic
topic_model.save(
    "lelkamel/chunks/bertopic_model",
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model="hugom123/bge-centralbank"
)
print("Modèle BERTopic sauvegardé !")

df_chunks_with_topics sauvegardé !
Modèle BERTopic sauvegardé !
